In [11]:
import numpy as np

In [1]:
from smve_lab.config import ARTIFACTS_DIR
from smve_lab.storage import load_separate, get_sparse, get_colbert

save_dir = ARTIFACTS_DIR / "embeddings" / "scifact_bgem3"
emb, ids = load_separate(save_dir, "docs") 

In [6]:
len(emb), len(ids)

(6, 5183)

In [7]:
query_emb, query_ids = load_separate(save_dir, "queries")

len(query_emb), len(query_ids), len(emb), len(ids)

(6, 1109, 6, 5183)

In [8]:
emb.keys()

dict_keys(['dense', 'sparse_indptr', 'sparse_ids', 'sparse_weights', 'colbert_flat', 'colbert_offsets'])

In [10]:
emb['colbert_flat']

memmap([[ 0.01834 , -0.00328 , -0.005455, ...,  0.02863 , -0.02145 ,
         -0.02669 ],
        [ 0.03198 , -0.0533  , -0.05887 , ...,  0.005726, -0.04916 ,
         -0.00393 ],
        [ 0.01936 , -0.010284, -0.02287 , ...,  0.01907 , -0.02412 ,
         -0.006474],
        ...,
        [ 0.009445,  0.05228 , -0.04114 , ...,  0.04654 , -0.002611,
         -0.0217  ],
        [ 0.02438 , -0.000596, -0.01888 , ...,  0.02649 ,  0.01744 ,
          0.00524 ],
        [ 0.02437 , -0.000636, -0.01888 , ...,  0.02649 ,  0.01752 ,
          0.005142]], shape=(1861078, 1024), dtype=float16)

In [1]:
import numpy as np
from pathlib import Path
from smve_lab.config import ARTIFACTS_DIR
from smve_lab.storage import load_separate, get_sparse, get_colbert

In [31]:
save_dir = ARTIFACTS_DIR / "embeddings" / "scifact_bgem3"
doc_emb, doc_ids = load_separate(save_dir, "docs", mmap_colbert=True)
query_emb, query_ids = load_separate(save_dir, "queries")

In [41]:
# total docs
len(doc_ids), type(doc_ids)

(5183, list)

In [ ]:
query1 = get_colbert(query_emb, 0) 
query1.shape

(14, 1024)

In [33]:
doc_vecs1 = get_colbert(doc_emb, 0)
doc_vecs1.shape

(447, 1024)

In [13]:
type(query_emb), type(query_ids), type(emb), type(ids)

(dict, list, dict, list)

In [15]:
query_emb.keys()

dict_keys(['dense', 'sparse_indptr', 'sparse_ids', 'sparse_weights', 'colbert_flat', 'colbert_offsets'])

In [16]:
emb.keys()

dict_keys(['dense', 'sparse_indptr', 'sparse_ids', 'sparse_weights', 'colbert_flat', 'colbert_offsets'])

In [ ]:
query1 = get_colbert(query_emb, 0) 
sim_scores = []
for i in range(len(doc_ids)):
    doc_vecs = get_colbert(doc_emb, i) #  447 x 1024
    #query1 = # [14 x 1024]
    S = query1 @ doc_vecs.T 
    max_sim = np.sum(np.max(S, axis=1))
    sim_scores.append((doc_ids[i], max_sim))


In [44]:
print(len(sim_scores))

5183


In [45]:
scores = np.array([score for _, score in sim_scores])
best_idx = np.argmax(scores)

best_doc_id, best_score = sim_scores[best_idx]

print(f"Best Index:  {best_idx}")
print(f"Best Doc ID: {best_doc_id}")
print(f"Best Score:  {best_score:.4f}")

Best Index:  4436
Best Doc ID: 35149431
Best Score:  6.3867


In [ ]:
import heapq

k = 5
topk = heapq.nlargest(k, sim_scores, key=lambda x: x[1])   
for doc_id, score in topk:
    print(doc_id, score)

35149431 6.387
4346436 6.324
20279166 6.17
45200347 6.168
4983 6.1


In [48]:
from smve_lab.scifact import load_scifact

corpus, queries, qrels, sf_doc_ids, doc_texts, sf_query_ids, query_texts = load_scifact()

doc_id_to_text = dict(zip(sf_doc_ids, doc_texts))
query_id_to_text = dict(zip(sf_query_ids, query_texts))


query_text = query_id_to_text[query_ids[0]]   # query_ids[0] is the id string for query1
print("QUERY:", query_text)

for doc_id, score in topk:
    print(f"[{score:.4f}] {doc_id}: {doc_id_to_text[doc_id][:200]}")

/Users/ujwaljibhkate/Projects/smve-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


QUERY: . 0-dimensional biomaterials lack inductive properties.
[6.3867] 35149431: P0 glycoprotein peptides 56–71 and 180–199 dose-dependently induce acute and chronic experimental autoimmune neuritis in Lewis rats associated with epitope spreading. Two synthetic peripheral nerve my
[6.3242] 4346436: Nonlinear Elasticity in Biological Gels. Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear elastic response, critical for the physiological funct
[6.1719] 20279166: Label-free DNA imaging in vivo with stimulated Raman scattering microscopy.. Label-free DNA imaging is highly desirable in biology and medicine to perform live imaging without affecting cell function 
[6.1680] 45200347: Resequencing studies of nonmodel organisms using closely related reference genomes: optimal experimental designs and bioinformatics approaches for population genomics.. Decreasing costs of next-genera
[6.1016] 4983: Microstructural development of human newbor